## Intrinsic Value Analysis Model Training

Intrinsic value is an estimation of a stock’s “fair value” based on Benjamin Graham’s formula. However, the formula ignores sector specific changes in valuations, cyclical business models which has yearly fluctuations in growth, and etc. often lending to incorrect assessments of good companies. Thus, a classification ML model which predicts by looking at failure points of Intrinsic valuations can make better judgments on what "fair value" actually looks like.

This model will use a Gaussian Mixture Model to make initial IV predictions, then look at points of failure and learn to veto the GMM's decisions for value trap cases.

#### Model Layers

Layer 1:
- calculates the "Graham” Intrinsic value
- accepts historical data from 2017-2021
- price, EPS, P/E constant at 7, avg. AAA corp bond yield at 4.4, expected growth, and AAA corp bond Yield during 2021
- outputs the Intrinsic Value Difference vs price (expressed as a percentage)

Layer 2:
- Gaussian Mixture Model (GMM), predicts sector relative valuation trends
- uses P/E ratio and Intrinsic Value Difference vs price
- outputs two natural clusters per sector "overvalued" and "undervalued"

Layer 3:
- Neural Network for classification
- accepts attributes like P/E ratio, Graham Intrinsic Values, and GMM prediction clusters
- accepts labels made from real price movement between 2021-2025
- acts as if predictions are being made in 2021 and checks using 2021-2025 price movement

#### Imports

In [50]:
from warnings import filterwarnings

import pandas as pd
import numpy as np
from sqlalchemy import exc, create_engine

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.mixture import GaussianMixture
from sklearn.neural_network import MLPClassifier

#### Loading data and Limited Pre-processing

In [54]:
server  = 'SERVER=localhost\\SQLEXPRESS'
db_name = 'DATABASE=stock_metrics'
driver  = 'DRIVER=ODBC Driver 17 for SQL Server'
params  = f"{driver};{server};{db_name};Trusted_Connection=yes;"
con_str = f"mssql+pyodbc:///?odbc_connect={params}"

# tables from the local sql database:
# stocks (ticker, stock_name, sector, price, earnings, pe_ratio, market_cap)
# growth_rates (ticker, start_value, end_value, growth_rate, corp_bond_yield)
# historic_valuations (ticker, valuation)
def load_data():
    filterwarnings("ignore", category=exc.SAWarning)
    engine = create_engine(con_str)
    query = """
    SELECT 
        s.ticker, s.sector, s.price, s.earnings, s.pe_ratio, 
        g.growth_rate, g.corp_bond_yield, h.valuation
    FROM dbo.stocks s
    JOIN dbo.growth_rates g ON s.ticker = g.ticker
    JOIN dbo.historic_valuations h ON s.ticker = h.ticker
    WHERE s.price > 0
    """
    df = pd.read_sql(query, engine)
    return df


stocks = load_data()
stocks['valuation'] = stocks['valuation'].map({
    'undervalued': 1, 'overvalued': 0
})
print(f"total stocks: {len(stocks)}")
print(f"total undervalued: {stocks['valuation'].value_counts().get(1, 0)}")
print(f"total overvalued: {stocks['valuation'].value_counts().get(0, 0)}")

total stocks: 851
total undervalued: 429
total overvalued: 422


#### Model Layer 1

In [55]:
# adds the intrinsic value difference as a percentage to the dataframe
# IV = (Earnings * (7 + 1 * Growth Rate) * 4.4) / AAA Corporate Bond Yield (2021)
# this will be an engineered feature for the GMM in the next layer
def calculate_iv_difference(row):
    growth = row['growth_rate'] * 100
    iv = (row['earnings'] * (7 + growth) * 4.4) / row['corp_bond_yield']
    return (iv - row['price']) / row['price']

stocks['iv_diff'] = stocks.apply(calculate_iv_difference, axis=1)
display(stocks.head(10))

,ticker,sector,price,earnings,pe_ratio,growth_rate,corp_bond_yield,valuation,iv_diff
0,A,Health Care,65.05,2.10,27.45,0.36,2.67,1,1.287609
1,AAL,Industrials,48.60,3.91,9.92,-0.22,2.67,1,-2.988718
2,AAP,Consumer Discretionary,109.63,6.19,19.54,0.08,2.67,0,0.395706
3,AAPL,Information Technology,155.15,9.20,16.86,0.56,2.67,1,5.156275
4,ABBV,Health Care,108.48,3.29,19.41,0.25,2.67,1,0.599328
5,ABT,Health Care,56.27,0.26,22.51,0.39,2.67,1,-0.649736
6,ACN,Information Technology,150.51,5.44,25.47,0.35,2.67,1,1.501637
7,ADBE,Information Technology,185.16,3.39,52.31,0.56,2.67,0,0.900792
8,ADI,Information Technology,82.68,2.11,17.67,0.29,2.67,1,0.514000
9,ADM,Consumer Staples,41.35,2.17,17.45,0.12,2.67,1,0.643158


#### Model Layer 2

In [56]:
analyzed_stocks = stocks.copy()
analyzed_stocks['gmm_label'] = np.nan

# applying GMM clustering to each sector separately
for sector in analyzed_stocks['sector'].unique():
    sector_data = analyzed_stocks[analyzed_stocks['sector'] == sector]
    if len(sector_data) < 2:
            continue
        
    gmm_features = sector_data[['pe_ratio', 'iv_diff']]
    scaled_features = StandardScaler().fit_transform(gmm_features)
    
    gmm = GaussianMixture(n_components=2, random_state=42)
    labels = gmm.fit_predict(scaled_features)
    
    cluster_0 = sector_data.iloc[labels == 0]['iv_diff'].mean() # overvalued
    cluster_1 = sector_data.iloc[labels == 1]['iv_diff'].mean() # undervalued
    if cluster_1 > cluster_0:
        final_labels = [1 if l == 1 else 0 for l in labels]
    else:
        final_labels = [1 if l == 0 else 0 for l in labels]
    analyzed_stocks.loc[
        analyzed_stocks['sector'] == sector, 'gmm_label'
    ] = final_labels
    
# convert gmm_label to int for easier analysis and use in later model layer
analyzed_stocks['gmm_label'] = analyzed_stocks['gmm_label'].astype(int)
display(analyzed_stocks.head(10))

,ticker,sector,price,earnings,pe_ratio,growth_rate,corp_bond_yield,valuation,iv_diff,gmm_label
0,A,Health Care,65.05,2.10,27.45,0.36,2.67,1,1.287609,1
1,AAL,Industrials,48.60,3.91,9.92,-0.22,2.67,1,-2.988718,1
2,AAP,Consumer Discretionary,109.63,6.19,19.54,0.08,2.67,0,0.395706,0
3,AAPL,Information Technology,155.15,9.20,16.86,0.56,2.67,1,5.156275,1
4,ABBV,Health Care,108.48,3.29,19.41,0.25,2.67,1,0.599328,1
5,ABT,Health Care,56.27,0.26,22.51,0.39,2.67,1,-0.649736,1
6,ACN,Information Technology,150.51,5.44,25.47,0.35,2.67,1,1.501637,1
7,ADBE,Information Technology,185.16,3.39,52.31,0.56,2.67,0,0.900792,0
8,ADI,Information Technology,82.68,2.11,17.67,0.29,2.67,1,0.514000,1
9,ADM,Consumer Staples,41.35,2.17,17.45,0.12,2.67,1,0.643158,1


#### Model Layer 3

In [92]:
classifier_features = ['pe_ratio', 'iv_diff', 'gmm_label']
X = analyzed_stocks[classifier_features]
y = analyzed_stocks['valuation']

X_scaled = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=100
)

nn_model = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),
    activation='relu', solver='adam',
    alpha=0.01, learning_rate_init=0.001,
    max_iter=3500, random_state=100
)
nn_model.fit(X_train, y_train)
predictions = nn_model.predict(X_test)

print("--- Evaluation with a Threshold ---")
print(classification_report(y_test, predictions))

--- Evaluation with a Threshold ---
              precision    recall  f1-score   support

           0       0.86      0.60      0.71        83
           1       0.71      0.91      0.80        88

    accuracy                           0.76       171
   macro avg       0.79      0.76      0.75       171
weighted avg       0.78      0.76      0.75       171

